In [2]:
import numpy as np

from numba import njit

from scipy.integrate import solve_ivp

from scipy.linalg import eigh

from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, root_mean_squared_error, accuracy_score

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

import fastplotlib as fpl

import optuna

# Init

In [3]:
steps = 20000

tau_steps = 1

transient_steps_henon = int(steps * 0.1)
transient_steps_reservoir = int(steps * 0.1)

total_steps = steps + transient_steps_henon + transient_steps_reservoir + tau_steps
total_steps_after_henon = steps + transient_steps_reservoir + tau_steps

test_size = 0.2
test_steps = int(steps * test_size)

t = np.arange(0, total_steps)

In [4]:
henon_dataset = np.zeros((total_steps, 2))

rng = np.random.default_rng(42)
henon_dataset[0] = rng.random(2)

a = 1.4
b = 0.3

In [5]:
@njit(fastmath=True, cache=True)
def henon_numba(steps, a=1.4, b=0.3, x0=0.0, y0=0.0):
    X = np.zeros(steps)
    Y = np.zeros(steps)
    X[0] = x0
    Y[0] = y0

    for i in range(1, steps):
        X[i] = 1 - a * X[i - 1] ** 2 + Y[i - 1]
        Y[i] = b * X[i - 1]

    return X, Y

In [6]:
henon_data_x, henon_data_y = henon_numba(total_steps)

henon_dataset = np.column_stack((henon_data_x, henon_data_y))
henon_dataset = henon_dataset[transient_steps_henon:]

In [7]:
henon_scaler = StandardScaler()
henon_train_scaled = henon_scaler.fit_transform(henon_dataset[:-test_steps])
henon_test_scaled = henon_scaler.transform(henon_dataset[-test_steps:])
henon_scaled = np.concatenate((henon_train_scaled, henon_test_scaled), axis=0)

In [8]:
def henon_plot(data_list, names=["Test", "Pred"], colors=["white", "magenta"]):
    fig = go.Figure()

    for i, data in enumerate(data_list):
        fig.add_trace(
            go.Scattergl(
                x=data[:, 0],
                y=data[:, 1],
                mode="markers",
                name=names[i] if names else f"Dataset {i+1}",
                marker=dict(color=colors[i % len(colors)], size=1),
            )
        )

    fig.update_layout(template="plotly_dark", title="Attractor Comparison")

    return fig

In [9]:
def r_2_plots_grid(actual_list, predicted_list, titles):
    fig = make_subplots(rows=1, cols=actual_list.shape[1], subplot_titles=titles)

    for i in range(actual_list.shape[1]):
        actual = actual_list[:, i]
        predicted = predicted_list[:, i]
        r_2 = r2_score(actual, predicted)
        col = i + 1

        fig.add_trace(
            go.Scatter(
                x=actual,
                y=predicted,
                mode="markers",
                name="Data",
                marker=dict(color="rgba(50, 50, 200, 0.5)", size=5),
            ),
            row=1,
            col=col,
        )

        min_val, max_val = min(actual.min(), predicted.min()), max(
            actual.max(), predicted.max()
        )
        fig.add_trace(
            go.Scatter(
                x=[min_val, max_val],
                y=[min_val, max_val],
                mode="lines",
                name="Ideal",
                line=dict(color="firebrick", dash="dash"),
            ),
            row=1,
            col=col,
        )

        fig.update_xaxes(title_text="Actual", row=1, col=col)
        fig.update_yaxes(title_text=f"Predicted (R²: {r_2:.4f})", row=1, col=col)

    fig.update_layout(showlegend=False, height=500, width=1000)
    return fig

In [10]:
def plot_grid(nodes_pos):
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=nodes_pos[:, 0],
            y=nodes_pos[:, 1],
            mode="markers",
            marker=dict(size=8, color="teal"),
        )
    )

    fig.update_layout(
        title="Hexagonal Lattice Distribution",
        xaxis=dict(title="X Position", scaleanchor="y", scaleratio=1),
        yaxis=dict(title="Y Position"),
        plot_bgcolor="white",
        width=700,
        height=700,
    )

    return fig


In [11]:
def weight_plot(weights):
    labels = [
        f"{'Pos' if i % 2 == 0 else 'Vel'} Node {i//2 + 1}" for i in range(len(weights))
    ]

    fig = go.Figure(
        data=[
            go.Bar(
                x=labels,
                y=weights,
                marker_color=np.where(weights >= 0, "royalblue", "firebrick"),
            )
        ]
    )

    fig.update_layout(
        title="Reservoir Node Contribution (Feature Weights)",
        xaxis_title="Spring/Mass Node",
        yaxis_title="Weight Value",
        template="plotly_white",
    )

    return fig

In [12]:
def spring_animation(
    disp,
    nodes_pos,
    connections_list,
    size=15,
    external=False,
    is_3d=False,
    frames_moved=5,
    max_frames=2000,
    animate=True,
    highlighted_nodes=None,
):
    steps = disp.shape[0]
    num_nodes = nodes_pos.shape[0]
    dims = nodes_pos.shape[1]

    disp_reshaped = disp.reshape(steps, num_nodes, dims)
    disp_3d = np.pad(disp_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")
    nodes_pos_3d = np.pad(nodes_pos, ((0, 0), (0, 3 - dims)), mode="constant")

    fig = (
        fpl.Figure(canvas="glfw" if external else "jupyter")
        if not is_3d
        else fpl.Figure(
            cameras="3d",
            controller_types="orbit",
            canvas="glfw" if external else "jupyter",
        )
    )

    node_colors = np.array(["magenta"] * num_nodes)
    if highlighted_nodes is not None:
        node_colors[highlighted_nodes] = "lime"

    coords = nodes_pos_3d + disp_3d[0]
    dots = fig[0, 0].add_scatter(
        data=coords.astype(np.float32), sizes=size, colors=node_colors
    )

    lines = [
        fig[0, 0].add_line(
            data=np.vstack([coords[int(row[0])], coords[int(row[1])]]).astype(
                np.float32
            ),
            thickness=2,
            colors="cyan",
        )
        for row in connections_list
    ]

    frame_tracker = 0
    test = True

    def update_springs(canvas):
        nonlocal frame_tracker, test
        # if not test:
        #     return
        # test = False
        frame_tracker = (frame_tracker + frames_moved) % steps

        if frame_tracker >= max_frames:
            canvas.clear_animations()

        coords = nodes_pos_3d + disp_3d[frame_tracker]
        dots.data = coords.astype(np.float32)

        for row, l in zip(connections_list, lines):
            src, dst = int(row[0]), int(row[1])
            l.data = np.vstack([coords[src], coords[dst]]).astype(np.float32)

    if animate:
        fig.add_animations(update_springs)
    return fig

# Calc Init

In [13]:
@njit(fastmath=True, cache=True)
def create_stiffness_matrix(node_positions, connections, k_vals):
    num_nodes = node_positions.shape[0]
    dims = node_positions.shape[1]
    K = np.zeros((num_nodes * dims, num_nodes * dims))

    for node_conn, k_val in zip(connections, k_vals):
        node_pos = node_positions[node_conn]
        diff_vec = node_pos[1] - node_pos[0]
        unit_dir = diff_vec / np.linalg.norm(diff_vec)
        sub_block = np.outer(unit_dir, unit_dir)

        idx1 = node_conn[0] * dims
        idx2 = node_conn[1] * dims

        K[idx1 : idx1 + 2, idx1 : idx1 + 2] += k_val * sub_block
        K[idx2 : idx2 + 2, idx2 : idx2 + 2] += k_val * sub_block
        K[idx1 : idx1 + 2, idx2 : idx2 + 2] += k_val * -sub_block
        K[idx2 : idx2 + 2, idx1 : idx1 + 2] += k_val * -sub_block
    return K

In [14]:
@njit(fastmath=True, cache=True)
def run_simulation(
    steps, dt, matrix_size, M_INV, C, U, initial_pos, connections_list, k_vals, constrained_nodes, constrained_values
):
    disp = np.zeros((steps, matrix_size))
    v = np.zeros((steps, matrix_size))
    acc = np.zeros(matrix_size)

    dims = initial_pos.shape[1]

    for i in range(1, steps):
        actual_pos = initial_pos + disp[i - 1].reshape(-1, dims)
        K = create_stiffness_matrix(actual_pos, connections_list, k_vals)
        for node in constrained_nodes:
            idx = node * dims
            K[idx, idx] += constrained_values
            K[idx + 1, idx + 1] += constrained_values

        acc = M_INV @ (-K @ disp[i - 1] - C @ v[i - 1] + U[i - 1])

        disp[i] = disp[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        acc_next = M_INV @ (-K @ disp[i] - C @ (v[i - 1] + acc * dt) + U[i])

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    return disp, v

# Single Hex

In [15]:
side_len = 1
x = np.array(
    [-side_len / 2, side_len / 2, side_len, side_len / 2, -side_len / 2, -side_len]
)
y = np.array(
    [
        0,
        0,
        side_len * np.sqrt(3) / 2,
        side_len * np.sqrt(3),
        side_len * np.sqrt(3),
        side_len * np.sqrt(3) / 2,
    ]
)
nodes_pos = np.column_stack((x, y))

plot_grid(nodes_pos).show()

In [16]:
rng = np.random.default_rng(42)

tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

node_m= rng.uniform(0.1, 0.3, size=num_nodes)
m_diag = np.repeat(node_m, dims)
m_inv_diag = 1.0 / m_diag
M = np.diag(m_diag)
M_INV = np.diag(m_inv_diag)

node_c = rng.uniform(0.05, 0.3, size=num_nodes)
c_diag = np.repeat(node_c, dims)
DAMP = np.diag(c_diag)

rng = np.random.default_rng(42)

U = np.zeros((steps + transient_steps_reservoir + tau_steps, matrix_size))
target_nodes = np.array([2, 5])
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
vectorized_force = np.zeros((U.shape[0], len(col_indices)))
vectorized_force[:, 0] = henon_scaled[:, 0]
vectorized_force[:, 2] = henon_scaled[:, 1]
U[:, col_indices] = vectorized_force

In [17]:
node_ids = np.arange(x.size)

src_nodes = node_ids
dst_nodes = np.roll(node_ids, 1)

rng = np.random.default_rng(42)
k_vals = rng.uniform(0.5, 8, size=src_nodes.shape[0])
connections_list = np.column_stack((src_nodes, dst_nodes))

In [21]:
displacement, velocity = run_simulation(
    steps + transient_steps_reservoir + tau_steps,
    0.01,
    matrix_size,
    M_INV,
    DAMP,
    U * 10,
    nodes_pos,
    connections_list,
    k_vals,
    [0, 1, 3, 4],
    100
)

X = np.column_stack((displacement, velocity))

In [22]:
X_delayed = X[:-tau_steps]
X_data = X_delayed[transient_steps_reservoir:]

x_scaler = StandardScaler()
X_train_scaled, X_test = (
    x_scaler.fit_transform(X_data[:-test_steps]),
    x_scaler.transform(X_data[-test_steps:]),
)

Y_train_scaled, Y_test_scaled = (
    henon_train_scaled[transient_steps_reservoir + tau_steps :],
    henon_test_scaled,
)

model = RidgeCV()
model.fit(X_train_scaled, Y_train_scaled)

Y_pred_scaled = model.predict(X_test)
Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)
Y_test = henon_scaler.inverse_transform(Y_test_scaled)

r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
print(f"{r_2:.4f}", f"{mse:.4f}")

0.0093 0.4682


In [23]:
weight_plot(np.linalg.norm(model.coef_, axis=0)).show()
henon_plot([Y_test, Y_pred]).show()
r_2_plots_grid(Y_test, Y_pred, ["Test", "Pred"]).show()
spring_animation(
    displacement,
    nodes_pos,
    connections_list,
    10,
    highlighted_nodes=[3, 4],
    external=True,
).show()

## Optuna

In [25]:
def hyper_param_input(input_force):
    displacement, velocity = run_simulation(
        steps + transient_steps_reservoir + tau_steps,
        0.01,
        matrix_size,
        M_INV,
        DAMP,
        U * input_force,
        nodes_pos,
        connections_list,
        k_vals,
        [0, 1, 3, 4],
        100,
    )
    X = np.column_stack((displacement, velocity))

    X_delayed = X[:-tau_steps]
    X_data = X_delayed[transient_steps_reservoir:]
    x_scaler = StandardScaler()
    X_train_scaled, X_test = (
        x_scaler.fit_transform(X_data[:-test_steps]),
        x_scaler.transform(X_data[-test_steps:]),
    )
    Y_train_scaled, Y_test_scaled = (
        henon_train_scaled[transient_steps_reservoir + tau_steps :],
        henon_test_scaled,
    )
    model = RidgeCV()
    model.fit(X_train_scaled, Y_train_scaled)
    Y_pred_scaled = model.predict(X_test)
    Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)
    Y_test = henon_scaler.inverse_transform(Y_test_scaled)

    return Y_test, Y_pred

In [29]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    input_force = trial.suggest_float("input_force", 0.1, 100.0)

    Y_test, Y_pred = hyper_param_input(input_force)

    r_2 = r2_score(Y_test, Y_pred)
    mse = root_mean_squared_error(Y_test, Y_pred)

    return r_2, mse


study = optuna.create_study(directions=["maximize", "minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=100, n_jobs=-1)

[Optuna] Processing Trial #999...

In [30]:
for trial in study.best_trials:
    print(f"Trial #{trial.number}")
    print(f"  Values: {trial.values}")
    print(f"  Params: {trial.params}")

Trial #362
  Values: [0.14996591664132486, 0.4461720105147471]
  Params: {'input_force': 0.12052768119849121}


In [31]:
Y_test, Y_pred = hyper_param_input(study.best_trials[0].params["input_force"])
weight_plot(np.linalg.norm(model.coef_, axis=0)).show()
henon_plot([Y_test, Y_pred]).show()
r_2_plots_grid(Y_test, Y_pred, ["Test", "Pred"]).show()
spring_animation(
    displacement,
    nodes_pos,
    connections_list,
    10,
    highlighted_nodes=target_nodes,
    external=True,
).show()